# Lab 04. 가설검정, 효과크기와 상관

이 Notebook은 두 집단 평균 차이와 두 수치 변수 관계를 분석한다.
p값만 보고 결론을 내리지 않고 원 단위 차이, 효과크기, 가정과 한계를 함께 보고한다.

Pearson 상관계수:

$$r=\frac{\sum(x_i-\bar{x})(y_i-\bar{y})}
{\sqrt{\sum(x_i-\bar{x})^2\sum(y_i-\bar{y})^2}}$$

In [ ]:
from pathlib import Path
import sys, subprocess

REPO_URL = "https://github.com/niko2204/bigdataservice.git"
if "google.colab" in sys.modules:
    ROOT = Path("/content/bigdataservice")
    if not ROOT.exists():
        subprocess.run(["git", "clone", "-q", REPO_URL, str(ROOT)], check=True)
else:
    candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
    ROOT = next((p.resolve() for p in candidates if (p / "src").exists()), None)
    if ROOT is None:
        raise FileNotFoundError("bigdataservice 저장소 루트에서 Notebook을 실행하세요.")

sys.path.insert(0, str(ROOT))
STUDENT_ID = "20260001"  # 반드시 본인 학번으로 변경
print("저장소:", ROOT)
print("실습 학번:", STUDENT_ID)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from src.education.personalized_data import make_student_dataset

df = make_student_dataset(STUDENT_ID).drop_duplicates()
for column in ["유동인구", "월임대료"]:
    df[column] = df.groupby("업종")[column].transform(lambda s: s.fillna(s.median()))
display(df.groupby("업종")["월매출"].agg(["count", "mean", "median", "std"]).round(1))

## 1. 분석 질문과 가설

질문: 개인 데이터에서 카페와 음식점의 평균 월매출이 다른가?

- H0: 카페와 음식점의 모집단 평균 월매출은 같다.
- H1: 두 모집단 평균 월매출은 다르다.
- 유의수준: alpha=0.05

두 집단 분산이 같다고 가정하지 않는 Welch t검정을 사용한다.

In [ ]:
cafe = df.loc[df["업종"] == "카페", "월매출"].dropna()
restaurant = df.loc[df["업종"] == "음식점", "월매출"].dropna()
t_stat, p_value = stats.ttest_ind(cafe, restaurant, equal_var=False)
mean_difference = cafe.mean() - restaurant.mean()

print("카페 n, 평균:", len(cafe), cafe.mean())
print("음식점 n, 평균:", len(restaurant), restaurant.mean())
print("평균차(카페-음식점):", mean_difference)
print("Welch t, p:", t_stat, p_value)

p<0.05이면 H0를 기각하지만 “H0가 틀릴 확률이 95%”라고 말하지 않는다.
p>=0.05도 두 집단이 같다는 증명이 아니라 현재 표본에서 차이를 확인할 증거가 충분하지 않다는 뜻이다.

## 2. 효과크기 Cohen d

$$d=\frac{\bar{x}_1-\bar{x}_2}{s_{pooled}}$$

효과크기는 차이를 표준편차 단위로 표현한다. 분야 맥락과 원 단위 차이를 함께 해석한다.

In [ ]:
def cohens_d(a, b):
    a, b = np.asarray(a), np.asarray(b)
    pooled_var = (
        (len(a)-1) * a.var(ddof=1) + (len(b)-1) * b.var(ddof=1)
    ) / (len(a) + len(b) - 2)
    return (a.mean() - b.mean()) / np.sqrt(pooled_var)

d = cohens_d(cafe, restaurant)
print("Cohen d:", d)

## 3. 완성 예제: 상관계수를 공식과 라이브러리로 비교

산점도를 먼저 확인한다. 이상값, 비선형 패턴과 업종별 집단이 섞인 구조를 살핀다.

In [ ]:
x = df["유동인구"].to_numpy()
y = df["월매출"].to_numpy()
numerator = ((x - x.mean()) * (y - y.mean())).sum()
denominator = np.sqrt(((x-x.mean())**2).sum() * ((y-y.mean())**2).sum())
r_manual = numerator / denominator
r_pandas = df["유동인구"].corr(df["월매출"])

print("직접 계산 r:", r_manual, "pandas r:", r_pandas)
plt.figure(figsize=(7, 4))
for category, group in df.groupby("업종"):
    plt.scatter(group["유동인구"], group["월매출"], alpha=.65, label=category)
plt.xlabel("유동인구")
plt.ylabel("월매출")
plt.legend()
plt.title("업종별 유동인구와 월매출")
plt.show()

## 4. 전체 상관과 집단 내 상관

전체 관계가 집단별 관계와 다를 수 있다. 집단 구성이 만든 관계인지 확인한다.

In [ ]:
correlations = (
    df.groupby("업종")
      .apply(lambda g: g["유동인구"].corr(g["월매출"]), include_groups=False)
      .rename("업종내_r")
)
print("전체 r:", r_pandas)
display(correlations)

## 5. 독립 연습

1. 주차장수가 중앙값 이상인 집단과 미만인 집단의 월매출을 비교한다.
2. 집단 정의, H0·H1, 검정 선택 이유, 표본 수, 평균차, p값, Cohen d를 표로 제시한다.
3. 월임대료–월매출의 Pearson r과 Spearman rho를 비교한다.
4. 관계에 영향을 줄 수 있는 제3의 변수 두 개를 제안한다.

In [ ]:
# TODO: 주차장 두 집단 비교
parking_cut = df["주차장수"].median()
high_parking = None
low_parking = None
test_result = None
effect_size = None

# TODO: Pearson과 Spearman 비교
correlation_comparison = None
display(correlation_comparison)

## 6. 자가점검

- [ ] H0와 H1을 분석 전에 작성했다.
- [ ] p값을 귀무가설이 참일 확률로 해석하지 않는다.
- [ ] 효과크기와 원 단위 평균차를 함께 보고했다.
- [ ] 상관관계를 인과관계로 표현하지 않았다.
- [ ] 전체 관계와 집단 내 관계를 비교했다.